In [ ]:
import os
import sys
import time
import logging
from pathlib import Path
import requests

_PROJECT_ROOT = Path.cwd().resolve()
if _PROJECT_ROOT.name == "notebooks":
    _PROJECT_ROOT = _PROJECT_ROOT.parent.parent
elif _PROJECT_ROOT.name == "podscan":
    _PROJECT_ROOT = _PROJECT_ROOT.parent
sys.path.insert(0, str(_PROJECT_ROOT))

from google_utils.google_sheet import GoogleSheetService
from data.constants import CREDENTIALS_FILE

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1pPF2clctk6NxAfI5AjgUJhasy8YrT1b_9XEf6MGbJe0/edit?gid=1458675702#gid=1458675702"

ENTITY_ID_COL_LETTER = "X"
ENTITY_COL_INDEX = 23  # 0-based index for column X
SKIP_SHEET = "List Info"

PODSCAN_API_BASE = "https://podscan.fm/api/v1"
PODSCAN_API_KEY = os.getenv("PODSCAN_API_KEY", "")
if not PODSCAN_API_KEY:
    logger.warning("PODSCAN_API_KEY not set. Set it before running the main loop.")

# Rate limiting (Podscan free tier: 5 req/min)
REQUESTS_PER_MINUTE = 5
API_DELAY_SECONDS = 60 / REQUESTS_PER_MINUTE  # 12s between calls
WRITE_BATCH_SIZE = 5
SKIP_IF_ALREADY_FILLED = True

In [ ]:
def fetch_guest_entity_id(episode_id: str, api_key: str) -> str | None:
    """
    Call GET /episodes/{episodeId}/entities?role=guest and return the first
    guest entity ID, or None if no guest is found / on error.
    """
    if not episode_id or not str(episode_id).strip():
        return None
    if not api_key or not str(api_key).strip():
        logger.error("Podscan API key is required")
        return None

    url = f"{PODSCAN_API_BASE}/episodes/{episode_id.strip()}/entities"
    headers = {"Authorization": f"Bearer {api_key}"}
    params = {"role": "guest"}

    def _do_request():
        return requests.get(url, headers=headers, params=params, timeout=30)

    try:
        resp = _do_request()

        if resp.status_code == 200:
            data = resp.json()
            guests = data.get("entities", {}).get("guests", [])
            if guests:
                return guests[0].get("id")
            logger.info(f"No guests found for episode {episode_id}")
            return None

        if resp.status_code == 401:
            logger.error("Podscan API: 401 Unauthorized - check API key")
            return None
        if resp.status_code == 404:
            logger.warning(f"Podscan API: 404 Not found for episode {episode_id}")
            return None
        if resp.status_code == 429:
            retry_after = 60
            logger.warning(f"Podscan API: 429 Rate limited. Waiting {retry_after}s before retry...")
            time.sleep(retry_after)
            resp = _do_request()
            if resp.status_code == 200:
                data = resp.json()
                guests = data.get("entities", {}).get("guests", [])
                if guests:
                    return guests[0].get("id")
                return None
            if resp.status_code == 429:
                logger.warning("Podscan API: 429 on retry - skipping")
                return None

        logger.warning(f"Podscan API: {resp.status_code} for episode {episode_id}: {resp.text[:200]}")
        return None
    except requests.RequestException as e:
        logger.warning(f"Podscan API network error for episode {episode_id}: {e}")
        return None

In [ ]:
sheet_service = GoogleSheetService(credentials_file=CREDENTIALS_FILE)
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)

success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
if success:
    sheets_to_process = [s for s in sheet_names if s != SKIP_SHEET]
    print(f"Spreadsheet ID: {spreadsheet_id}")
    print(f"All sheets: {sheet_names}")
    print(f"Sheets to process (excluding '{SKIP_SHEET}'): {sheets_to_process}")
else:
    print(f"Error listing sheets: {sheet_names}")

In [ ]:
def filter_unique_episodes(data: list, episode_id_col: str, status_col: str) -> list:
    """
    Keep unique episode_id; when duplicates exist with Pending and Completed, prefer Completed.
    """
    if not data:
        return []
    headers = data[0]
    try:
        episode_id_idx = headers.index(episode_id_col)
        status_idx = headers.index(status_col)
    except ValueError:
        return data

    result_map = {}
    for row in data[1:]:
        episode_id = row[episode_id_idx] if episode_id_idx < len(row) else ""
        if not episode_id or not str(episode_id).strip():
            continue
        status = row[status_idx] if status_idx < len(row) else ""
        if episode_id not in result_map:
            result_map[episode_id] = row
        else:
            existing_status = result_map[episode_id][status_idx] if status_idx < len(result_map[episode_id]) else ""
            if existing_status == "Pending" and status == "Completed":
                result_map[episode_id] = row

    return [headers] + list(result_map.values())


success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
if not success or not isinstance(sheet_names, list):
    print(f"Cannot list sheets: {sheet_names}")
else:
    sheets_to_process = [s for s in sheet_names if s != SKIP_SHEET]
    for sheet_name in sheets_to_process:
        success, data = sheet_service.get_sheet_values(spreadsheet_id, f"'{sheet_name}'!A:Z")
        if not success:
            print(f"[{sheet_name}] Error: {data}")
            continue
        if not data:
            print(f"[{sheet_name}] No data, skipping")
            continue

        headers = data[0]
        if "episode_id" not in headers or "analysis_status" not in headers:
            print(f"[{sheet_name}] Missing episode_id or analysis_status columns, skipping")
            continue

        rows_before = len(data) - 1
        filtered = filter_unique_episodes(data, "episode_id", "analysis_status")
        rows_after = len(filtered) - 1

        success, msg = sheet_service.clear_and_rewrite_sheet(spreadsheet_id, sheet_name, filtered)
        if success:
            print(f"[{sheet_name}] {rows_before} -> {rows_after} rows ({rows_before - rows_after} duplicates removed)")
        else:
            print(f"[{sheet_name}] Error: {msg}")

In [ ]:
def flush_updates(updates: list) -> int:
    if not updates:
        return 0
    sheet_service.service.spreadsheets().values().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body={"valueInputOption": "RAW", "data": list(updates)},
    ).execute()
    count = len(updates)
    updates.clear()
    return count


def run_guest_entity_enrichment():
    if not PODSCAN_API_KEY:
        logger.error("Set PODSCAN_API_KEY environment variable and re-run")
        return

    success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
    if not success or not isinstance(sheet_names, list):
        logger.error(f"Cannot list sheets: {sheet_names}")
        return

    sheets_to_process = [s for s in sheet_names if s != SKIP_SHEET]
    logger.info(f"Processing {len(sheets_to_process)} sheets (excluding '{SKIP_SHEET}')")

    for sheet_name in sheets_to_process:
        logger.info("=" * 60)
        logger.info(f"Sheet: {sheet_name}")
        logger.info("=" * 60)

        try:
            success, data = sheet_service.get_sheet_values(
                spreadsheet_id, f"'{sheet_name}'!A:Z"
            )
            if not success or not data:
                logger.info("  No data or error, skipping.")
                continue

            headers = data[0]
            try:
                episode_id_idx = headers.index("episode_id")
                has_guests_idx = headers.index("has_guests")
            except ValueError as e:
                logger.warning(f"  Missing column: {e}, skipping.")
                continue

            updates = []
            total_written = 0
            processed = 0
            found = 0
            failed = 0
            skipped_filled = 0
            skipped_no_guests = 0
            skipped_empty = 0

            for i, row in enumerate(data[1:], start=2):
                episode_id = (row[episode_id_idx] if episode_id_idx < len(row) else "").strip()
                existing_entity = (row[ENTITY_COL_INDEX] if ENTITY_COL_INDEX < len(row) else "").strip()
                has_guests = (row[has_guests_idx] if has_guests_idx < len(row) else "").strip()

                if SKIP_IF_ALREADY_FILLED and existing_entity:
                    skipped_filled += 1
                    continue
                if has_guests.upper() != "TRUE":
                    skipped_no_guests += 1
                    continue
                if not episode_id:
                    skipped_empty += 1
                    continue

                processed += 1
                entity_id = fetch_guest_entity_id(episode_id, PODSCAN_API_KEY)

                if entity_id:
                    found += 1
                    updates.append({
                        "range": f"'{sheet_name}'!{ENTITY_ID_COL_LETTER}{i}",
                        "values": [[entity_id]],
                    })
                    logger.info(f"  [row {i}] {episode_id} -> {entity_id}")
                else:
                    failed += 1
                    logger.info(f"  [row {i}] {episode_id} -> no guest entity found")

                if len(updates) >= WRITE_BATCH_SIZE:
                    written = flush_updates(updates)
                    total_written += written
                    logger.info(f"  ** Flushed batch of {written} (total written: {total_written})")

                time.sleep(API_DELAY_SECONDS)

            if updates:
                written = flush_updates(updates)
                total_written += written
                logger.info(f"  ** Flushed final batch of {written} (total written: {total_written})")

            logger.info(
                f"  Summary: processed={processed} | found={found} | failed={failed} | "
                f"skipped_filled={skipped_filled} | skipped_no_guests={skipped_no_guests} | "
                f"skipped_empty={skipped_empty} | written={total_written}"
            )

        except Exception as e:
            logger.exception(f"  ERROR on sheet '{sheet_name}': {e}")

    logger.info("All sheets processed.")


run_guest_entity_enrichment()